In [10]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

# Configurar dispositivo en CPU (esencial para comparar PTQ en PyTorch)
device = torch.device("cpu")

# 1. Preparar datos (subconjunto rápido de CIFAR-10)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
# Usamos 500 imágenes para que la evaluación en CPU tome pocos segundos
test_subset = torch.utils.data.Subset(testset, range(500))
testloader = torch.utils.data.DataLoader(test_subset, batch_size=32, shuffle=False)

# 2. Cargar modelo base
model_fp32 = resnet18(weights=ResNet18_Weights.DEFAULT)
model_fp32.fc = nn.Linear(model_fp32.fc.in_features, 10) # Adaptar a 10 clases
model_fp32.to(device).eval()

# TODO: Implementar funciones de medición (tamaño, latencia, accuracy)
# TODO: Implementar Pruning (Parte 2)
# TODO: Implementar Quantization (Parte 3)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

### Funciones de Medición (Tamaño, Latencia y Accuracy)

In [11]:
def medir_tamano(model, filepath="temp_model.p"):
    """Calcula el tamaño del modelo en disco en Megabytes (MB)."""
    torch.save(model.state_dict(), filepath)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    os.remove(filepath)
    return size_mb

def medir_latencia(model, dataloader, device, num_runs=3):
    """Mide la latencia promedio de inferencia sobre el dataloader en CPU."""
    model.to(device)
    model.eval()
    tiempos = []
    with torch.no_grad():
        for _ in range(num_runs):
            inicio = time.time()
            for inputs, _ in dataloader:
                inputs = inputs.to(device)
                _ = model(inputs)
            tiempos.append(time.time() - inicio)
    return sum(tiempos) / len(tiempos)

def medir_accuracy(model, dataloader, device):
    """Calcula la precisión (Accuracy %) en el dataset de prueba."""
    model.to(device)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

### Implementación de Pruning (Paso 2)

In [12]:
import copy

# Creamos una copia del modelo original para aplicar la poda
model_pruned = copy.deepcopy(model_fp32)

# Identificamos todas las capas Conv2d para aplicar la poda globalmente
parametros_a_podar = []
for modulo in model_pruned.modules():
    if isinstance(modulo, nn.Conv2d):
        parametros_a_podar.append((modulo, 'weight'))

# Aplicamos Global Unstructured L1 Pruning (podamos el 40% de las conexiones más débiles)
prune.global_unstructured(
    parametros_a_podar,
    pruning_method=prune.L1Unstructured,
    amount=0.40,
)

# Calculamos el porcentaje de sparsity (ceros) en los pesos convolucionales
total_elementos = 0
total_ceros = 0
for modulo, nombre_param in parametros_a_podar:
    peso = getattr(modulo, nombre_param)
    total_elementos += peso.nelement()
    total_ceros += torch.sum(peso == 0).item()

print(f"Sparsity global en capas de convolución: {100. * total_ceros / total_elementos:.2f}%")

Sparsity global en capas de convolución: 40.00%


### Evaluación y Comparativa de Métricas (Original vs. Podado)

In [13]:
# Evaluar modelo original FP32
tam_fp32 = medir_tamano(model_fp32)
lat_fp32 = medir_latencia(model_fp32, testloader, device)
acc_fp32 = medir_accuracy(model_fp32, testloader, device)

# Evaluar modelo podado (Pruned)
tam_pruned = medir_tamano(model_pruned)
lat_pruned = medir_latencia(model_pruned, testloader, device)
acc_pruned = medir_accuracy(model_pruned, testloader, device)

# Mostrar tabla de resultados
print(f"| Métrica             | Modelo Original FP32 | Modelo Podado (40%) |")
print(f"|---------------------|----------------------|----------------------|")
print(f"| Tamaño en disco     | {tam_fp32:.2f} MB           | {tam_pruned:.2f} MB           |")
print(f"| Latencia (Inferencia)| {lat_fp32:.4f} s          | {lat_pruned:.4f} s          |")
print(f"| Accuracy (CIFAR-10) | {acc_fp32:.2f}%             | {acc_pruned:.2f}%             |")

| Métrica             | Modelo Original FP32 | Modelo Podado (40%) |
|---------------------|----------------------|----------------------|
| Tamaño en disco     | 42.73 MB           | 85.33 MB           |
| Latencia (Inferencia)| 47.8396 s          | 48.1131 s          |
| Accuracy (CIFAR-10) | 8.60%             | 8.80%             |


### Implementación de Cuantización Dinámica (Paso 3)

Aplicamos la cuantización dinámica convirtiendo los pesos de tipo `nn.Linear` a enteros de 8 bits (INT8).

In [14]:
import torch.quantization

# Creamos una copia del modelo original para cuantizar
model_quantized = torch.quantization.quantize_dynamic(
    model_fp32,             # Modelo base
    {nn.Linear},            # Capas a cuantizar dinámicamente
    dtype=torch.qint8       # Tipo de destino (INT8)
)

print("¡Modelo cuantizado dinámicamente con éxito!")

¡Modelo cuantizado dinámicamente con éxito!


/tmp/ipykernel_2850/1176587948.py:4: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_quantized = torch.quantization.quantize_dynamic(


### Evaluación y Comparación Final (Paso 4 - Análisis)

Evaluamos el modelo cuantizado y mostramos la comparativa completa de métricas de rendimiento y eficiencia en disco.

In [15]:
# Evaluar modelo cuantizado
tam_quant = medir_tamano(model_quantized)
lat_quant = medir_latencia(model_quantized, testloader, device)
acc_quant = medir_accuracy(model_quantized, testloader, device)

# Mostrar tabla comparativa completa
print(f"| Métrica             | Modelo FP32 Original | Modelo Podado (40%) | Modelo Cuantizado (INT8) |")
print(f"|---------------------|----------------------|----------------------|---------------------------|")
print(f"| Tamaño en disco     | {tam_fp32:.2f} MB           | {tam_pruned:.2f} MB           | {tam_quant:.2f} MB                |")
print(f"| Latencia (Inferencia)| {lat_fp32:.4f} s          | {lat_pruned:.4f} s          | {lat_quant:.4f} s               |")
print(f"| Accuracy (CIFAR-10) | {acc_fp32:.2f}%             | {acc_pruned:.2f}%             | {acc_quant:.2f}%                  |")

| Métrica             | Modelo FP32 Original | Modelo Podado (40%) | Modelo Cuantizado (INT8) |
|---------------------|----------------------|----------------------|---------------------------|
| Tamaño en disco     | 42.73 MB           | 85.33 MB           | 42.71 MB                |
| Latencia (Inferencia)| 47.8396 s          | 48.1131 s          | 48.2008 s               |
| Accuracy (CIFAR-10) | 8.60%             | 8.80%             | 8.40%                  |
